# vLLM Benchmark — Code-Generation Workload

All implementation lives in `vllm_bench/src/`; this notebook only configures, launches, and saves runs.

1. **Concurrency × Workload sweep** — three code-gen profiles across many concurrency levels
2. **ISL sweep** — TTFT vs. input length (fixed OSL, fixed concurrency)
3. **OSL sweep** — TPOT / E2E vs. output length (fixed ISL, fixed concurrency)


## 1. Setup: Drive, dependencies, project import

In [ ]:
# Mount Drive and pip-install pinned deps. Restarts runtime once after install.
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
PROJECT_ROOT = '/content/drive/MyDrive/vllm_bench'   # repo root on Drive
SENTINEL     = f'{PROJECT_ROOT}/.install_ok_v3'
os.makedirs(PROJECT_ROOT, exist_ok=True)

PINS = {
    'numpy': '1.26.4', 'pandas': '2.2.2', 'pyarrow': '18.1.0',
    'torch': '2.5.1', 'torchvision': '0.20.1', 'torchaudio': '2.5.1',
    'xformers': '0.0.28.post3', 'transformers': '4.48.2',
    'tokenizers': '0.21.0', 'vllm': '0.7.3',
}

def _check():
    import importlib.metadata as im
    bad = []
    for pkg, ver in PINS.items():
        try:
            if im.version(pkg) != ver: bad.append(pkg)
        except im.PackageNotFoundError:
            bad.append(pkg)
    return bad

def _run(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end='')
    p.wait(); return p.returncode

if not os.path.exists(SENTINEL) or _check():
    print('Installing pinned deps...')
    _run('pip uninstall -y ' + ' '.join(PINS.keys()) + ' -q')
    args = ' '.join(f'{k}=={v}' for k, v in PINS.items())
    if _run(f'pip install {args} nest_asyncio aiohttp huggingface_hub --no-cache-dir -q') != 0:
        raise RuntimeError('pip install failed')
    open(SENTINEL, 'w').write('ok')
    print('Restarting runtime — re-run this cell after restart'); os.kill(os.getpid(), 9)

# Make `from src...` work from notebooks/
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import nest_asyncio; nest_asyncio.apply()
import torch, vllm, transformers, numpy, pandas
print(f'torch {torch.__version__} | vllm {vllm.__version__} | '
      f'transformers {transformers.__version__} | numpy {numpy.__version__}')


## 2. Auto-select model and prepare paths

In [ ]:
from src.config import Paths, auto_select_model, detect_vram_gb, SERVER_PORT

paths = Paths.colab_default(PROJECT_ROOT).ensure()
vram  = detect_vram_gb()
model_cfg = auto_select_model(vram).with_local_path(paths.model_dir)

print(f'GPU VRAM:    {vram:.1f} GB')
print(f'Model:       {model_cfg.model_id}')
print(f'Quant:       {model_cfg.quantization or "None (FP16)"}')
print(f'Max len:     {model_cfg.max_model_len}')
print(f'Local path:  {model_cfg.local_path}')
print(f'Results dir: {paths.results_dir}')


## 3. Download model (if not cached on Drive) and start vLLM server

In [ ]:
from src.server import download_model, start_server, verify_inference

download_model(model_cfg)
server_proc = start_server(model_cfg, port=SERVER_PORT, wait_timeout=600)
print('\nSanity check:', verify_inference(SERVER_PORT)[:120], '...')


## 4. Workload generator

In [ ]:
from src.workload import WorkloadGenerator, WORKLOAD_PROFILES

gen = WorkloadGenerator(model_cfg.local_path)
SERVER_URL = f'http://localhost:{SERVER_PORT}'

print('Workload profiles:')
for name, p in WORKLOAD_PROFILES.items():
    print(f'  {name:22s}  ISL{p["ISL_range"]}  OSL{p["OSL_range"]}  '
          f'weight={p["weight"]}  TTFT_SLO={p["TTFT_SLO"]}s')

# Token-precision smoke test
for tgt in [128, 1024, 4096]:
    actual = len(gen.tokenizer.encode(gen.generate_prompt(tgt, seed=0),
                                      add_special_tokens=False))
    print(f'  prompt target={tgt:5d}  actual={actual:5d}  diff={actual-tgt:+d}')


## 5. Experiment 1 — Concurrency × Workload

In [ ]:
import asyncio
from datetime import datetime
from src.experiments import run_concurrency_sweep
from src.storage import save_run
from src.plots import plot_baseline

CONCURRENCY_LEVELS = [1, 2, 4, 8, 16, 24, 32, 48, 64, 96, 128, 200]
PROFILES_TO_TEST   = ['inline_completion', 'code_explanation', 'function_generation']

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
metrics, traces = asyncio.get_event_loop().run_until_complete(
    run_concurrency_sweep(
        gen, PROFILES_TO_TEST, CONCURRENCY_LEVELS, SERVER_URL,
        model_name=model_cfg.model_name,
        quantization=model_cfg.quantization or 'none',
    )
)
df_metrics, df_traces, _ = save_run(metrics, traces, paths.results_dir,
                                    tag='standard_baseline', timestamp=ts)
plot_baseline(df_metrics, PROFILES_TO_TEST, model_cfg.model_name,
              paths.results_dir, ts)


## 6. Experiment 2 — ISL Sweep

In [ ]:
import asyncio
from datetime import datetime
from src.experiments import run_isl_sweep
from src.storage import save_run
from src.plots import plot_isl_sweep

ISL_VALUES = [128, 256, 512, 1024, 1536, 2048, 2560,
              3072, 3584, 4096, 5120, 6144, 7680]
FIXED_OSL  = 1024
FIXED_CONC = 8
N_PER_ISL  = 100

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
isl_metrics, isl_traces = asyncio.get_event_loop().run_until_complete(
    run_isl_sweep(gen, ISL_VALUES, SERVER_URL,
                  fixed_osl=FIXED_OSL, fixed_concurrency=FIXED_CONC,
                  n_per_isl=N_PER_ISL)
)
isl_df, _, _ = save_run(isl_metrics, isl_traces, paths.results_dir,
                        tag='isl_sweep', timestamp=ts)
plot_isl_sweep(isl_df, FIXED_OSL, FIXED_CONC, N_PER_ISL,
               paths.results_dir, ts)


## 7. Experiment 3 — OSL Sweep

In [ ]:
import asyncio
from datetime import datetime
from src.experiments import run_osl_sweep
from src.storage import save_run
from src.plots import plot_osl_sweep

OSL_VALUES = [50, 100, 250, 500, 1000, 2000, 3000, 4000, 6000]
FIXED_ISL  = 512
FIXED_CONC = 8
N_PER_OSL  = 200

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
osl_metrics, osl_traces = asyncio.get_event_loop().run_until_complete(
    run_osl_sweep(gen, OSL_VALUES, SERVER_URL,
                  fixed_isl=FIXED_ISL, fixed_concurrency=FIXED_CONC,
                  n_per_osl=N_PER_OSL)
)
_, osl_traces_df, _ = save_run(osl_metrics, osl_traces, paths.results_dir,
                                tag='osl_sweep', timestamp=ts)
plot_osl_sweep(osl_traces_df, FIXED_ISL, FIXED_CONC, N_PER_OSL,
               paths.results_dir, ts)


## 8. SLO sensitivity analysis

In [ ]:
from src.storage import load_latest
from src.metrics import (DEFAULT_SLO_CANDIDATES, compute_slo_table,
                         slo_comparison_matrix, slo_information_summary)

_, df_traces = load_latest(paths.results_dir, tag='standard_baseline')

for name, slo_map in DEFAULT_SLO_CANDIDATES.items():
    print(f'\n── SLO candidate: {name} ──')
    summary = compute_slo_table(df_traces, slo_map)
    summary['ttft_p50_ms'] = (summary['ttft_p50'] * 1000).round(0).astype(int)
    summary['ttft_p99_ms'] = (summary['ttft_p99'] * 1000).round(0).astype(int)
    summary['slo_rate_%']  = (summary['slo_rate'] * 100).round(1)
    print(summary[['profile', 'concurrency', 'n',
                   'ttft_p50_ms', 'ttft_p99_ms', 'slo_rate_%']].to_string(index=False))

print('\n── Comparison matrix (% met) ──')
cmp_df = slo_comparison_matrix(df_traces, DEFAULT_SLO_CANDIDATES)
print(cmp_df.to_string(index=False))

print('\n── Information / discriminative power ──')
print(slo_information_summary(cmp_df, DEFAULT_SLO_CANDIDATES).to_string(index=False))


## 9. Cleanup

In [ ]:
from src.server import stop_server
stop_server(server_proc)
print(f'Results in: {paths.results_dir}')
!ls -lh {paths.results_dir}
